In [ ]:
from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def find_parentheses_pairs(s):
    stack = []  # Stack to keep track of '(' positions
    pairs = {}  # Dictionary to store pairs of indices (key is '(' or ')', value is the paired parenthesis index)
    
    # Iterate through the string with index
    for i, char in enumerate(s):
        if char == '(':
            stack.append(i)  # Push the index of '(' onto the stack
        elif char == ')':
            if stack:  # Ensure stack is not empty
                open_index = stack.pop()  # Pop the index of the matching '('
                pairs[open_index] = i  # Store the pair (open_index, close_index)
                pairs[i] = open_index  # Store the reverse pair (close_index, open_index)
    
    return pairs

def get_outermost_paired_parenthesis(pairs, position):
    # If position is an opening parenthesis, we find its outermost pair
    if position in pairs:
        # Return the paired parenthesis
        return pairs[position]
    else:
        return None  # If no matching parenthesis found for the given position

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments



file_paths = [
    'Feature_guide_target_complete_struct_only_bhah_NM_002794_4_KD_reduced_feature_model.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  # No need to write anything, just open and close the file to delete its content

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('RfxCas13d_mmc4_filtered_full_guide_sequences_NM_002794_4.txt', 'r')    # Input crRNA sequences with AsCas12a direct repeat
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('RfxCas13d_mmc4_filtered_target_sequences_noPAM_NM_002794_4.txt', 'r')    # Input target sequences with AsCas12a direct repeat
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('RfxCas13d_mmc4_filtered_target_sequences_noPAM_NM_002794_4.txt', 'r')    # Input target sequences with AsCas12a direct repeat
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

for i in range (0, len(guide_array)):

    # Initialize struct_prob_array
    struct_prob_array = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Initialize struct_prob_array
    struct_prob_array_guide = []
    struct_prob_array_target = []

    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    subopt_structures_guide = subopt(strands=guide, energy_gap=0.01, model=my_model_RNA)  # The energy gap of 5.68 kcal/mol refers to minimal probablity of 0.01%

    # Compute ensemble energy before and after hybridization
    partition_function_guide_bh = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide_bh = partition_function_guide_bh[1]
    partition_function_target_bh = pfunc(strands=[target_truncated, RNA_reverse_complement(target_truncated)], model=my_model_RNA)
    ensemble_energy_target_bh = partition_function_target_bh[1]
    
    partition_function_hybrid_ah = pfunc(strands=[guide, target_truncated], model=my_model_RNA)
    ensemble_energy_hybrid_ah = partition_function_hybrid_ah[1]
    partition_function_target_ah = pfunc(strands=RNA_reverse_complement(target_truncated), model=my_model_RNA)
    ensemble_energy_target_ah = partition_function_target_ah[1]

    partition_function_target_ssRNA = pfunc(strands=target_truncated, model=my_model_RNA)
    ensemble_energy_target_ssRNA = partition_function_target_ah[1]

    # Compute suboptimal structures and energy
    subopt_structures_guide_bh = subopt(strands=guide, energy_gap=0.01, model=my_model_RNA)  # The energy gap of 5.68 kcal/mol refers to minimal probablity of 0.01%
    subopt_structures_target_bh = subopt(strands=[target_truncated, RNA_reverse_complement(target_truncated)], energy_gap=0.01, model=my_model_RNA)
    subopt_structures_hybrid_ah = subopt(strands=[guide, target_truncated], energy_gap=0.01, model=my_model_RNA)
    subopt_structures_target_ssRNA = subopt(strands=target_truncated, energy_gap=0.01, model=my_model_RNA)

    '''
    print(subopt_structures_guide_bh[0].structure)
    print(subopt_structures_target_bh[0].structure)
    print(subopt_structures_hybrid_ah[0].structure)
    print()
    '''
    
    # Compute the probability for each suboptimal structure using Boltzmann equilibrium probability distribution (ViennaRNA package)
    struct_prob_array_unit = []
    kT = RNA.exp_param().kT / 1000.
    prob_sub_guide = math.exp((ensemble_energy_guide - subopt_structures_guide[0].energy) / kT)
    pairs_guide = find_parentheses_pairs(str(subopt_structures_guide[0].structure))
    
    # Guide structure before hybridization
    spacer_struct_bh = str(subopt_structures_guide_bh[0].structure)[36:66]
    for j in range (0, 30):
        
        try:
            if spacer_struct_bh[j] == '.':
                to_be_added = ([1, 0, 0])
            elif spacer_struct_bh[j] == '(':
                to_be_added = ([0, 1, 0])
            elif spacer_struct_bh[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
                
        except IndexError:
            to_be_added = ([0, 0, 0])

        struct_prob_array_unit.append(to_be_added)
        
        
    target_ssRNA_struct_bh = (str(subopt_structures_target_ssRNA[0].structure))
    for j in range (0, 30):
        
        try:
            if target_ssRNA_struct_bh[j] == '.':
                to_be_added = ([1, 0, 0])
            elif target_ssRNA_struct_bh[j] == '(':
                to_be_added = ([0, 1, 0])
            elif target_ssRNA_struct_bh[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_prob_array_unit.append(to_be_added)

         
    # Hybrid structure after hybridization 
    hybrid_ah_1 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[0]
    spacer_struct_ah = hybrid_ah_1[36:66]

    for j in range (0, 30):
        
        try:
            if spacer_struct_ah[j] == '.':
                to_be_added = ([1, 0, 0])
            elif spacer_struct_ah[j] == '(':
                to_be_added = ([0, 1, 0])
            elif spacer_struct_ah[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_prob_array_unit.append(to_be_added)

            
    struct_prob_array_to_append = [struct_prob_array_unit]
    struct_prob_array.append(struct_prob_array_to_append)

    struct_prob_array = list(itertools.chain.from_iterable(struct_prob_array))
    
    # Open a file in write mode
    with open('Feature_guide_target_complete_struct_only_bhah_NM_002794_4_KD_reduced_feature_model.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_prob_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

